# LSTM Baseline vs VAE (λ=0) vs Constrained VAE (λ=0.5)


In [ ]:
import os
import torch

from tokenizer import SMILESTokenizer
from model import Encoder, Decoder, VAE, PropertyPredictor
from generate import ConstrainedGenerator
from lstm_baseline import SMILESLanguageModel
from evaluate import compute_metrics, print_metrics

# paths and hyperparameters — must match what was used in SMILE_VAE.py / train_lstm.py
VOCAB_PATH      = "vocab.json"
CKPT_PATH       = "checkpoints/vae_best.pt"
LSTM_CKPT_PATH  = "checkpoints_lstm/lstm_best.pt"
LATENT_DIM  = 256
EMBED_DIM   = 128
HIDDEN_DIM  = 512
NUM_LAYERS  = 2
DROPOUT     = 0.1
TEMPERATURE = 0.8
GA_STEPS    = 50
GA_LR       = 0.01
BETA_ANCHOR = 0.01
N_MOLECULES = 200

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# load tokenizer and VAE from checkpoint
tok = SMILESTokenizer()
tok.load(VOCAB_PATH)

encoder  = Encoder(tok.vocab_size, EMBED_DIM, HIDDEN_DIM, LATENT_DIM, NUM_LAYERS, DROPOUT)
decoder  = Decoder(tok.vocab_size, EMBED_DIM, HIDDEN_DIM, LATENT_DIM, NUM_LAYERS, DROPOUT)
vae      = VAE(encoder, decoder).to(device)
pred_qed = PropertyPredictor(LATENT_DIM).to(device)
pred_sa  = PropertyPredictor(LATENT_DIM).to(device)

ckpt = torch.load(CKPT_PATH, map_location=device)
vae.load_state_dict(ckpt["vae"])
pred_qed.load_state_dict(ckpt["pred_qed"])
pred_sa.load_state_dict(ckpt["pred_sa"])
print(f"Loaded VAE checkpoint from epoch {ckpt['epoch']}")

In [ ]:
# load LSTM baseline from checkpoint
lstm = SMILESLanguageModel(tok.vocab_size, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(device)
lstm_ckpt = torch.load(LSTM_CKPT_PATH, map_location=device)
lstm.load_state_dict(lstm_ckpt["model"])
print(f"Loaded LSTM checkpoint from epoch {lstm_ckpt['epoch']}")

In [ ]:
import json
import matplotlib.pyplot as plt

with open("checkpoints/loss_history.json") as f:
    vae_history = json.load(f)

vae_epochs = range(1, len(vae_history["val_loss"]) + 1)

lstm_history = None
if os.path.exists("checkpoints_lstm/loss_history.json"):
    with open("checkpoints_lstm/loss_history.json") as f:
        lstm_history = json.load(f)

n_plots = 3 if lstm_history else 2
fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 4))

axes[0].plot(vae_epochs, vae_history["train_recon"], label="train recon")
axes[0].set_title("VAE Reconstruction Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(vae_epochs, vae_history["train_kl"],  label="train KL")
axes[1].plot(vae_epochs, vae_history["val_loss"],  label="val total", linestyle="--")
axes[1].set_title("VAE KL Loss & Val Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()

if lstm_history:
    lstm_epochs = range(1, len(lstm_history["val_loss"]) + 1)
    axes[2].plot(lstm_epochs, lstm_history["train_loss"], label="train")
    axes[2].plot(lstm_epochs, lstm_history["val_loss"],   label="val", linestyle="--")
    axes[2].set_title("LSTM Loss")
    axes[2].set_xlabel("Epoch")
    axes[2].legend()
else:
    print("LSTM loss history not found — run train_lstm.py to completion to see this plot.")

plt.tight_layout()
plt.savefig("loss_curve.png", dpi=150)
plt.show()

In [ ]:
# generate with VAE baseline (λ=0 — gradient ascent on QED only, no SA penalty)
gen_baseline = ConstrainedGenerator(vae, pred_qed, pred_sa, tok, lambda_=0.0, beta=BETA_ANCHOR, device=device)
smiles_baseline = gen_baseline.generate(N_MOLECULES, n_steps=GA_STEPS, ga_lr=GA_LR, temperature=TEMPERATURE)

print("VAE baseline (λ=0) samples:")
for s in smiles_baseline[:5]:
    print(" ", s)

In [ ]:
# generate with constrained VAE (λ=0.5 — balance QED and synthesis cost)
gen_constrained = ConstrainedGenerator(vae, pred_qed, pred_sa, tok, lambda_=0.5, beta=BETA_ANCHOR, device=device)
smiles_constrained = gen_constrained.generate(N_MOLECULES, n_steps=GA_STEPS, ga_lr=GA_LR, temperature=TEMPERATURE)

print("Constrained VAE (λ=0.5) samples:")
for s in smiles_constrained[:5]:
    print(" ", s)

In [ ]:
# generate with LSTM baseline (pure language model — no latent space, no optimization)
smiles_lstm = lstm.sample(N_MOLECULES, tok, temperature=TEMPERATURE, device=device)

print("LSTM baseline samples:")
for s in smiles_lstm[:5]:
    print(" ", s)

In [ ]:
# metrics — LSTM uses lambda_=0.5 for reward so all models are compared on same criterion
m_lstm = compute_metrics(smiles_lstm,        reference_set=set(), lambda_=0.5)
m0     = compute_metrics(smiles_baseline,    reference_set=set(), lambda_=0.5)
m05    = compute_metrics(smiles_constrained, reference_set=set(), lambda_=0.5)

print(f"{'metric':<15}  {'LSTM baseline':>14}  {'VAE (λ=0)':>12}  {'VAE (λ=0.5)':>14}")
print("-" * 62)
for k in m0:
    print(f"{k:<15}  {m_lstm[k]:>14.4f}  {m0[k]:>12.4f}  {m05[k]:>14.4f}")

literature review papers:
- Cyclic KL annealing: https://arxiv.org/abs/1903.10145 
- VAE: https://arxiv.org/abs/1312.6114
- SMILES VAE: https://arxiv.org/abs/1610.02415
